# 🧪 Headless Cloud Runner Smoke Test
This notebook verifies headless execution on **GitHub Actions** (2 vCPU, 7 GB RAM) using **Papermill**.
It inspects host hardware, performs numeric computation, plots results with Matplotlib, and exports artifacts.


In [ ]:
# Parameters (Papermill injects overrides below this cell)
n_samples = 500
noise_level = 0.12
experiment_title = "GitHub-Actions-Headless-Runner"
save_plot = True


In [ ]:
import os, sys, platform, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

output_dir = os.environ.get("OUTPUT_DIR", "output_results")
os.makedirs(output_dir, exist_ok=True)

print("=" * 60)
print(f"🖥️  Host System:    {platform.system()} {platform.release()} ({platform.machine()})")
print(f"🐍 Python Version: {sys.version.split()[0]}")
print(f"⚙️  CPU Count:      {os.cpu_count()} logical cores")
print(f"📁 Output Dir:     {os.path.abspath(output_dir)}")
print(f"🏷️  Experiment:     {experiment_title}")
print("=" * 60)


In [ ]:
# Generate synthetic signal
np.random.seed(42)
x = np.linspace(0, 4 * np.pi, n_samples)
true_signal = np.sin(x) + 0.5 * np.cos(2 * x)
noise = np.random.normal(0, noise_level, size=n_samples)
y = true_signal + noise

# Fit polynomial
coeffs = np.polyfit(x, y, deg=7)
poly_fit = np.polyval(coeffs, x)
rmse = float(np.sqrt(np.mean((poly_fit - y) ** 2)))

print(f"✓ Generated {n_samples} points with noise_level={noise_level}")
print(f"✓ Fitted 7th-order polynomial with RMSE: {rmse:.5f}")


In [ ]:
# Visualization
plt.figure(figsize=(10, 5), dpi=150)
plt.scatter(x, y, s=12, alpha=0.45, label="Observed Noisy Data", color="#1f77b4")
plt.plot(x, true_signal, "--", color="#2ca02c", linewidth=2.0, label="True Signal")
plt.plot(x, poly_fit, "-", color="#d62728", linewidth=2.5, label=f"Fit (RMSE={rmse:.4f})")

plt.title(f"{experiment_title} (N={n_samples})", fontsize=14, fontweight="bold")
plt.xlabel("X (radians)")
plt.ylabel("Signal Amplitude")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")
plt.tight_layout()

plot_path = os.path.join(output_dir, "smoke_test_plot.png")
if save_plot:
    plt.savefig(plot_path)
    print(f"✓ Saved visualization to {plot_path}")
plt.close()


In [ ]:
# Export CSV and JSON metrics
df = pd.DataFrame({
    "x": x[:50],
    "true_signal": true_signal[:50],
    "observed_y": y[:50],
    "fitted_y": poly_fit[:50]
})
csv_path = os.path.join(output_dir, "predictions.csv")
df.to_csv(csv_path, index=False)

metrics = {
    "experiment_title": experiment_title,
    "n_samples": n_samples,
    "noise_level": noise_level,
    "rmse": rmse,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "status": "SUCCESS"
}
json_path = os.path.join(output_dir, "metrics.json")
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"✓ Saved predictions sample to {csv_path}")
print(f"✓ Saved metrics to {json_path}")


In [ ]:
# Verification checks
assert os.path.exists(plot_path), f"Plot missing: {plot_path}"
assert os.path.exists(csv_path), f"CSV missing: {csv_path}"
assert os.path.exists(json_path), f"JSON metrics missing: {json_path}"
assert os.path.getsize(plot_path) > 1000, "Plot file too small"
assert os.path.getsize(csv_path) > 100, "CSV file too small"

print("\n" + "=" * 60)
print("🎉 [SUCCESS] All smoke test verification assertions passed!")
print("=" * 60)
